# 19 - Generate LLM Context from Well Logs

**Part B - Integrated & Advanced Workflows**

This lesson turns LAS data into compact, auditable JSON context packages organized by analysis category, well, formation, and lithology group. It uses only public `petrophys` APIs and bundled sample data, so it runs without machine-specific paths.

The generated JSON is model-neutral: use it with any LLM client, retrieval system, or review workflow.

## Learning goals

- Keep every user-controlled path in one configuration cell.
- Generate category-first context JSON from one or more LAS files.
- Split context by formation and contiguous lithology group.
- Check failures, inspect provenance, and validate the output before using it.
- Build a model-neutral request object without coupling the library to an LLM vendor.

## What the exporter produces

For each lithology group, the exporter writes separate views for hydrocarbon, saturation, permeability, porosity, numerical statistics, and lithology analysis. Each JSON file records the source well, formation interval, input logs, computed logs, statistics, analysis results, data completeness, and bad-hole QC.

Raw depth-by-depth traces are intentionally excluded by default. This keeps context smaller while retaining the statistics and provenance needed for interpretation and review.

In [1]:
from collections import Counter
from pathlib import Path
import json

import pandas as pd

import petrophys
from petrophys.reporting import (
    export_reference_json_work_package_by_category_formation_lithology,
    write_reference_curve_name_mapping,
)
from petrophys.workflow import DynamicWorkflowConfig

print("petrophys version:", petrophys.__version__)

petrophys version: 0.1.0


## 1. Configure paths once

All generated files go below `RUN_ROOT`. For this tutorial, `petrophys.sample(...)` resolves the packaged LAS and formation table regardless of where the package is installed.

In [2]:
# Change only this cell when adapting the lesson to another project.
WELL_NAME = "30_6-5"
RUN_ROOT = Path.cwd() / "outputs" / "llm_context_demo"
OUTPUT_DIR = RUN_ROOT / "context"
TOPS_FILE = RUN_ROOT / "inputs" / "formation_tops.xlsx"
MINIMUM_GROUP_THICKNESS = 2.0

# Bundled, portable tutorial inputs.
LAS_FILE = Path(petrophys.sample(WELL_NAME))
FORMATION_SOURCE = Path(petrophys.sample("formations_30_6-5"))
INPUT_DIR = LAS_FILE.parent

TOPS_FILE.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Interpretation parameters. The 30/6-5 header prints RW = 0.70 ohm.m with no
# temperature basis; taken at face value it wets out the entire well. The
# operator's evaluation reconciles to 0.070 ohm.m at reservoir temperature
# (lesson 09 back-calculates it; case study 1 cross-checks it against the
# NPD fact page), so we pass that pair explicitly. Every field left None still
# auto-fills from the LAS header block - explicit values simply win.
WORKFLOW_CONFIG = DynamicWorkflowConfig(rw=0.070, rw_meas_temp_f=230.0)

print("LAS:       ", LAS_FILE.name, "(bundled sample)")
print("Tops CSV:  ", FORMATION_SOURCE.name, "(bundled sample)")
print("Run root:  ", RUN_ROOT.relative_to(Path.cwd()))
print("Overrides: ", {"rw": WORKFLOW_CONFIG.rw, "rw_meas_temp_f": WORKFLOW_CONFIG.rw_meas_temp_f})

LAS:        30_6-5.las (bundled sample)
Tops CSV:   formations_30_6-5.csv (bundled sample)
Run root:   outputs\llm_context_demo
Overrides:  {'rw': 0.07, 'rw_meas_temp_f': 230.0}


For your own data, replace `LAS_FILE`, `FORMATION_SOURCE`, and `RUN_ROOT` in the cell above. The exporter expects a tops workbook with the columns `well`, `formation`, and `md_top`. An optional lithology-mapping workbook may also be supplied; when it is omitted, the library uses its built-in lithology normalization.

## 2. Prepare the formation-tops workbook

The exporter's workbook schema is deliberately small - `well`, `formation`,
`md_top` - with each formation's bottom implied by the next top. That schema
cannot represent a **nested** stratigraphy: the bundled table wraps groups
around their member formations (`BAAT GP.` 2841-2958 m contains Ness and
Etive), and feeding both levels in would truncate every group at its first
member's top and mislabel the intervals the LLM context is built on.

So we keep only the **leaf intervals** - rows that do not contain another
row - which tile the well contiguously and convert cleanly to the top-only
schema. (Lesson 11 covers the group/member hierarchy in depth.)

In [3]:
formations = pd.read_csv(FORMATION_SOURCE)
required_source_columns = {"formation_name", "top_depth", "bottom_depth"}
missing = required_source_columns.difference(formations.columns)
if missing:
    raise ValueError(f"Formation source is missing columns: {sorted(missing)}")

def is_leaf(row) -> bool:
    """True when no other interval sits strictly inside this one."""
    inside = (
        (formations["top_depth"] >= row.top_depth)
        & (formations["bottom_depth"] <= row.bottom_depth)
        & ~(
            (formations["top_depth"] == row.top_depth)
            & (formations["bottom_depth"] == row.bottom_depth)
        )
    )
    return not bool(inside.any())

leaves = formations[[is_leaf(r) for r in formations.itertuples()]].reset_index(drop=True)
print(f"{len(formations)} source rows -> {len(leaves)} contiguous leaf intervals")

tops = pd.DataFrame(
    {
        "well": WELL_NAME,
        "formation": leaves["formation_name"],
        "md_top": leaves["top_depth"],
    }
)
tops.to_excel(TOPS_FILE, index=False)
display(tops)
print("Wrote:", TOPS_FILE.relative_to(Path.cwd()))

20 source rows -> 16 contiguous leaf intervals


,well,formation,md_top
0,30_6-5,NORDLAND GP.,181.0
1,30_6-5,Utsira Fm.,726.0
2,30_6-5,HORDALAND GP.,879.0
3,30_6-5,Balder Fm.,1908.0
4,30_6-5,Sele Fm.,2023.0
5,30_6-5,Lista Fm.,2123.0
6,30_6-5,SHETLAND GP.,2203.0
7,30_6-5,CROMER KNOLL GP.,2408.0
8,30_6-5,Draupne Fm.,2452.0
9,30_6-5,Heather Fm.,2593.0


Wrote: outputs\llm_context_demo\inputs\formation_tops.xlsx


## 3. Generate category-first JSON context

The dynamic workflow standardizes the LAS, computes valid interpretations from the available curves, applies QC, slices the well by formation, groups contiguous lithology intervals, and writes one JSON document per analysis category and group.

In [4]:
results = export_reference_json_work_package_by_category_formation_lithology(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    tops_file=TOPS_FILE,
    pattern=LAS_FILE.name,
    recursive=False,
    minimum_group_thickness=MINIMUM_GROUP_THICKNESS,
    well_names=[WELL_NAME],
    lithology_mapping_file=None,
    workflow_config=WORKFLOW_CONFIG,
)
mapping_path = write_reference_curve_name_mapping(OUTPUT_DIR)

print("Generated records:", len(results))
print("Curve mapping:    ", Path(mapping_path).relative_to(Path.cwd()))

Generated records: 852
Curve mapping:     outputs\llm_context_demo\context\curve_name_mapping.json


## 4. Fail loudly and summarize the run

A context directory is not evidence of a successful run. Check every returned result and stop before sending incomplete output downstream.

In [5]:
failures = [item for item in results if not item.success]
if failures:
    details = "\n".join(
        f"- {Path(item.source_file).name}: {' | '.join(item.errors)}"
        for item in failures
    )
    raise RuntimeError(f"Context generation failed:\n{details}")

context_files = sorted(
    path
    for path in OUTPUT_DIR.rglob("*.json")
    if path.name != "curve_name_mapping.json"
)
category_counts = Counter(path.relative_to(OUTPUT_DIR).parts[0] for path in context_files)

summary = pd.DataFrame(
    {
        "category": list(category_counts),
        "json_files": list(category_counts.values()),
    }
).sort_values("category", ignore_index=True)
display(summary)
print("Successful exports:", len(results))
print("JSON context files:", len(context_files))

,category,json_files
0,hydrocarbon,142
1,lithology,142
2,numerical,142
3,permeability,142
4,porosity,142
5,saturation,142


Successful exports: 852
JSON context files: 852


In [6]:
# Show a compact, relative file listing instead of machine-specific paths.
for path in context_files[:12]:
    print(path.relative_to(RUN_ROOT))
if len(context_files) > 12:
    print(f"... and {len(context_files) - 12} more JSON files")

context\hydrocarbon\30_6-5\AMUNDSEN FM\AMUNDSEN FM._SILT_GROUP_2_hydrocarbon.json
context\hydrocarbon\30_6-5\AMUNDSEN FM\AMUNDSEN FM._SILT_GROUP_4_hydrocarbon.json
context\hydrocarbon\30_6-5\AMUNDSEN FM\AMUNDSEN FM._SILT_GROUP_5_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._EXTRUSIVE_GROUP_10_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._EXTRUSIVE_GROUP_11_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._EXTRUSIVE_GROUP_1_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._EXTRUSIVE_GROUP_3_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._EXTRUSIVE_GROUP_4_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._EXTRUSIVE_GROUP_5_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._EXTRUSIVE_GROUP_6_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._EXTRUSIVE_GROUP_7_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._SILT_GROUP_1_hydrocarbon.json
... and 84

## 5. Inspect one context document

Choose the category that matches the question being asked. Here we open a hydrocarbon context and inspect only high-signal metadata before using the complete payload.

In [7]:
hydrocarbon_files = [
    path for path in context_files
    if path.relative_to(OUTPUT_DIR).parts[0] == "hydrocarbon"
]
context_path = hydrocarbon_files[0] if hydrocarbon_files else context_files[0]
payload = json.loads(context_path.read_text(encoding="utf-8"))
analysis = payload["analysis_results"]

overview = pd.Series(
    {
        "file": str(context_path.relative_to(RUN_ROOT)),
        "well": analysis["well_name"],
        "formation": analysis["formation_name"],
        "lithology_group": analysis["lithology group"],
        "lithology": analysis["lithology"],
        "input_log_count": len(analysis["input_logs_present"]),
        "derived_log_count": len(analysis["derived_logs_computed"]),
        "raw_traces_included": payload["include_raw_traces_in_context"],
    }
)
display(overview.to_frame("value"))

,value
file,context\hydrocarbon\30_6-5\AMUNDSEN FM\AMUNDSE...
well,30_6-5
formation,AMUNDSEN FM.
lithology_group,SILT_GROUP_2
lithology,SILT
input_log_count,14
derived_log_count,48
raw_traces_included,False


In [8]:
# Basic schema and provenance checks before an LLM sees the document.
required_top_level = {
    "formation_info",
    "analysis_results",
    "raw_log_traces_in_segment",
    "include_raw_traces_in_context",
}
required_analysis = {
    "well_name",
    "formation_name",
    "statistics",
    "analysis_summary",
    "data_completeness",
    "input_logs_present",
    "derived_logs_computed",
    "BADHOLE_QC_CHECK",
}

assert required_top_level.issubset(payload)
assert required_analysis.issubset(analysis)
assert analysis["well_name"]
assert payload["formation_info"]["thickness"] >= 0
print("Schema and provenance checks passed.")

Schema and provenance checks passed.


### Physics sanity, not just schema

A context package can be schema-perfect and physically wrong. The classic
failure on this very well: use the header's `RW = 0.70` at face value and the
whole well silently reads wet - the JSON would then confidently describe the
Etive pay sand as water-bearing. Before any of this reaches an LLM, probe an
interval whose answer is known.

In [9]:
etive_sand = [
    path for path in context_files
    if path.relative_to(OUTPUT_DIR).parts[0] == "saturation"
    and any("etive" in part.lower() for part in path.parts)
    and "SAND" in path.name
]
assert etive_sand, "No Etive sand saturation context found - check the tops preparation"
probe = json.loads(etive_sand[0].read_text(encoding="utf-8"))
stats = probe["analysis_results"]["statistics"]
sw_mean = stats["SW_overall"]["arithmetic_mean"]
rw_mean = stats["RW_overall"]["arithmetic_mean"]

print(f"{etive_sand[0].name}")
print(f"  SW mean: {sw_mean:.3f}   RW (temperature-corrected) mean: {rw_mean:.3f} ohm.m")

assert 0.04 < rw_mean < 0.15, (
    "Rw basis looks wrong - compare WORKFLOW_CONFIG against the LAS header block"
)
assert sw_mean < 0.85, (
    "The Etive sand reads wet: a mis-based Rw silently invalidates the whole context"
)
print("Physics sanity passed: the flagship sand is not spuriously wet.")

ETIVE FM._CROSSBEDDED SAND_GROUP_1_saturation.json
  SW mean: 0.342   RW (temperature-corrected) mean: 0.072 ohm.m
Physics sanity passed: the flagship sand is not spuriously wet.


## 6. Build a model-neutral LLM request

Keep measured inputs, computed outputs, warnings, and missing-data statements intact. Ask the model to distinguish observation from inference and to avoid inventing values. The object below can be serialized for any LLM SDK or stored in a retrieval index.

In [10]:
llm_request = {
    "task": (
        "Summarize reservoir quality and hydrocarbon evidence for this interval. "
        "State data limitations and QC warnings."
    ),
    "instructions": [
        "Use only the supplied context.",
        "Separate measured logs from computed interpretation curves.",
        "Do not infer missing values.",
        "Report the well, formation, lithology group, and depth interval.",
        "Treat bad-hole flags and data-completeness warnings as interpretation limits.",
    ],
    "petrophysical_context": payload,
}

serialized_request = json.dumps(llm_request, ensure_ascii=False)
print("Request characters:", len(serialized_request))
print("Ready for an LLM client or retrieval pipeline.")

Request characters: 23283
Ready for an LLM client or retrieval pipeline.


## CLI equivalent

The same category-first exporter is available as a command-line tool. Inputs are explicit; no personal or repository-specific directories are assumed.

```bash
python scripts/generate_category_context_json_by_formation_lithology.py \
  --input-dir path/to/las \
  --tops-file path/to/formation_tops.xlsx \
  --output-dir outputs/llm_context \
  --well WELL_NAME
```

Use `--lithology-mapping-file path/to/mapping.xlsx` only when vendor codes need a project mapping. Use `--legacy-benchmark-compat` only to reproduce the frozen historical benchmark; it is not the recommended mode for new interpretations.

## Recap

- One configuration cell controls the input and output locations.
- The dynamic workflow creates category-specific, formation- and lithology-aware JSON.
- Every generated package carries provenance, completeness, statistics, and QC.
- Nested tops are reduced to leaf intervals before the top-only workbook schema.
- Explicit workflow parameters win over LAS-header values, so documented
  reconciliations (like this well's Rw basis) survive header-driven runs.
- Validation covers schema *and* physics before context is passed to an LLM.
- The output is model-neutral and does not require an LLM dependency inside `petrophys`.